In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import json

sys.path.insert(0, os.path.join(os.path.abspath(os.pardir), "src"))
from molearn.data import PDBData
from molearn.models.foldingnet import AutoEncoder
from molearn.analysis import MolearnAnalysis
from molearn.analysis.plot import *

import torch

We first create a directory to save the figures produced during analysis.

In [ ]:
fig_dir = Path('analysis_foldingnet_basic')
if fig_dir.exists() is False:
    fig_dir.mkdir(parents=True, exist_ok=True)

We now load three datasets for model evaluation:

1. **Training data (open state)**: MurD open conformation simulations
2. **Training data (closed state)**: MurD closed conformation simulations  
3. **Test data (transition)**: MurD closed → open transition trajectory

The test dataset contains intermediate conformational states that the model has never seen during training. Evaluating on this dataset allows us to assess how well the model generalizes to unseen conformations.

**Note**: Data Standardization

All datasets must be standardized using the **same mean and std** values that were used to standardize the combined training data. We load these statistics from `data_statistics.json` and pass them to the `prepare_dataset()` method to ensure consistent normalization across all datasets.

In [ ]:
atoms = ["N", "CA", "C", "O", 'CB']
mean = json.load(open("data_statistics.json"))["mean"]
std = json.load(open("data_statistics.json"))["std"]

In [ ]:
data_closed = PDBData(
    filename=["./data/MurD_closed.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_closed.prepare_dataset(mean=mean, std=std)

In [ ]:
data_open = PDBData(
    filename=["./data/MurD_open.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_open.prepare_dataset(mean=mean, std=std)

In [ ]:
data_test = PDBData(
    filename=["./data/MurD_closed_apo.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_test.prepare_dataset(mean=mean, std=std)

Load trained model checkpoint. Initialize MolearnAnalysis class and link to datasets and model. If the checkpoint file doesn't exist, run [cb_foldingnet_basic.py](cb_foldingnet_basic.py)

In [ ]:
checkpoint_file = next((Path('foldingnet_checkpoints')).glob('checkpoint_converged.ckpt'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
net = AutoEncoder(**checkpoint['network_kwargs'])
net.load_state_dict(checkpoint['model_state_dict'])
if torch.cuda.is_available():
    net.to(device)

In [ ]:
MA = MolearnAnalysis()
MA.set_network(net)
MA.batch_size = 16
MA.processes = 32

MA.set_dataset(data=data_open, key="train_open")
MA.set_dataset(data=data_closed, key="train_closed")
MA.set_dataset(data=data_test, key="test_trans")

### Model evaluation

`MolearnAnalysis` class ships with a variety of methods for computing quality metrics (reconstruction error, DOPE scores, Ramachandran statistics, bond lengths, chirality, etc.) on datasets and their autoencoder-decoded counterparts. There are also methods that compute metrics for structures grid sampled from the latent space and reconstructed.

Meanwhile, there is a bunch of plotting functions available in [`molearn/analysis/plot.py`](../src/molearn/analysis/plot.py) that visualize these metrics quickly. Of course you can custimize your own plots with the data produced by `MolearnAnalysis`.

The following cells show some examples of plots you can make.

---

Calculate and plot the reconstruction RMSD for datasets.

In [ ]:
rmsd_train_open = MA.get_error('train_open')
rmsd_train_closed = MA.get_error('train_closed')
rmsd_test_trans = MA.get_error('test_trans')

rmsd_plot_data = [
    ("train_open", "test_trans", "Open vs Transition", "train", "test"),
    ("train_closed", "test_trans", "Closed vs Transition", "train", "test"),
]

plot_rmsd_hist(
    MA,
    plot_data=rmsd_plot_data,
    fname=fig_dir / "rmsd_violin.png",
    dpi=150,
)

Calculate and plot the bondlengths for datasets.

In [ ]:
bondlength_train_open = MA.get_bondlengths('train_open')
bondlength_train_closed = MA.get_bondlengths('train_closed')
bondlength_test_trans = MA.get_bondlengths('test_trans')

bondlength_plot_data = [
    ("train_open", "Train Open", "#1f77b4"),
    ("train_closed", "Train Closed", "#ff7f0e"),
    ("test_trans", "Test Transition", "#2ca02c"),
]

plot_bondlength_hist(
    MA,
    plot_data=bondlength_plot_data,
    bins=60,
    wkdir=fig_dir,
    dpi=150,
)

Plot the number of inverted beta-carbons to assess stereochemical quality. 

This metric identifies chirality inversions at CA atoms, which are common structural artifacts in generative models. This analysis only works when 'CB' atoms are included in the atom selection during model training.

**Note:** The metric calculation can be performed internally by the plotting function, similar to `plot_bondlength_hist()` and `plot_rmsd_hist()` above.

In [ ]:
inversion_plot_data = [('train_open', 'train_open', "#1f77b4"),
                       ('train_closed', 'train_closed',"#ff7f0e"),
                       ('test_trans', 'test_trans', "#2ca02c")]

plot_inversion_hist(MA, 
                    plot_data = inversion_plot_data,
                    fname = fig_dir/'inversion_hist.pdf')

Calculate and visualize the DOPE (Discrete Optimized Protein Energy) score across the latent space.

DOPE is a statistical potential derived from known protein structures that estimates structural quality. By computing DOPE scores across a grid sampling of the latent space, we can create an energy landscape that reveals:
- **Low-energy regions** (favorable conformations similar to native structures)
- **High-energy regions** (potentially unrealistic or strained conformations)
- **Energy barriers** between different conformational states

This landscape is useful for identifying the most physically plausible paths between conformational states and understanding which regions of latent space correspond to well-formed protein structures.

In [ ]:
# We first setup a latent grid whose boundaries encompass all datasets.
if "grid" not in MA._encoded:
    grid_key = MA.setup_grid(samples=75)
    print(f"Latent grid '{grid_key}' initialised with {MA.n_samples} samples per axis.")
else:
    grid_key = "grid"
    print("Re-using previously initialised latent grid.")

# Next, we compute the DOPE surface over the latent grid.
dope_surface, xvals, yvals = MA.scan_dope(refine=True)
surface_clip = np.percentile(dope_surface, 80)

dope_plot_data = [
    ("train_open", "Train Open", "#FDBFCA", "scatter"),
    ("train_closed", "Train Closed", "#AFC2DC", "scatter"),
    ("test_trans", "Test Transition", "#7FB069", "scatter"),
]

plot_dope_surface(
    MA,
    refine=True,
    truncate_at=surface_clip,
    plot_data=dope_plot_data,
    cmap="viridis",
    fname=fig_dir / "dope_surface_refined.png",
    bbox_inches="tight",
)

`MolearnAnalysis.scan_custom` lets us evaluate any user-defined scalar metric across the latent grid. The method decodes each grid sample, applies the provided function to Cartesian coordinates (shape `(1, N, 3)` after rescaling), and stores the resulting surface.

In [ ]:
def radius_of_gyration(coords):
    """Compute the root-mean-square distance of atoms from their centroid."""
    coords = np.asarray(coords).reshape(-1, 3)
    centroid = coords.mean(axis=0)
    return np.sqrt(((coords - centroid) ** 2).sum(axis=1).mean())

rog_surface, xaxis, yaxis = MA.scan_custom(
    radius_of_gyration,
    params=(),
    key="radius_of_gyration",
)

### Interpolation and Generation

An important function of Molearn is to predict protein transition intermediates when trained with the end states of this transition. This can be done by generating a sequence of conformations reconstructed from samples that interpolate between two end states. 

The [`molearn/analysis/path.py`](../src/molearn/analysis/path.py) contains two methods for getting paths between two end points: oversample and A*. And then `MolearnAnalysis` provides methods that reconstruct 3D Cartisian coordinates from latent vectors and write conformations to PDB files. 

Following is an example of generating a path between the MurD open and the MurD closed states.

---

As a demonstration, we randomly sample one datapoints from the open and closed state training set. We calculate the path connecting them using linear interpolation and A* algorithm (with DOPE surface) respectively.

In [ ]:
encoded_open = MA.get_encoded("train_open").cpu().numpy()
encoded_closed = MA.get_encoded("train_closed").cpu().numpy()

rng = np.random.default_rng()
open_idx = int(rng.integers(0, encoded_open.shape[0]))
closed_idx = int(rng.integers(0, encoded_closed.shape[0]))

start_latent = encoded_open[open_idx]
end_latent = encoded_closed[closed_idx]

print(f"Selected open latent index: {open_idx}")
print(f"Selected closed latent index: {closed_idx}")

In [ ]:
from molearn.analysis.path import get_path, oversample, get_point_index

# Linear interpolation path using oversample
latent_path_linear = oversample(
    np.asarray([start_latent, end_latent])
)

# A* path using the DOPE surface
# First, we need to convert the latent coordinates to grid indices
start_idx = get_point_index(start_latent, xvals, yvals)
end_idx = get_point_index(end_latent, xvals, yvals)

# Use A* to find the best path between start and end
latent_path_astar = get_path(
    start_idx,
    end_idx,
    dope_surface,
    xvals,
    yvals,
)[0]

print(f"Linear path length: {len(latent_path_linear)}")
print(f"A* path length: {len(latent_path_astar)}")

Let's show the paths on the DOPE surface. 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot the DOPE surface
im = ax.contourf(xvals, yvals, dope_surface, levels=50, cmap='viridis', vmax=surface_clip)
plt.colorbar(im, ax=ax, label='DOPE Score')

# Plot the linear interpolation path
ax.plot(latent_path_linear[:, 0], latent_path_linear[:, 1], 
        'r-', linewidth=4, label='Linear Interpolation', alpha=0.8)

# Plot the A* path
ax.plot(latent_path_astar[:, 0], latent_path_astar[:, 1], 
        'c-', linewidth=4, label='A* Path', alpha=0.8)

# Plot the training data points
encoded_open_2d = MA.get_encoded("train_open").cpu().numpy()
encoded_closed_2d = MA.get_encoded("train_closed").cpu().numpy()
ax.scatter(encoded_open_2d[:, 0], encoded_open_2d[:, 1], 
           c='pink', s=20, alpha=0.5, label='Train Open')
ax.scatter(encoded_closed_2d[:, 0], encoded_closed_2d[:, 1], 
           c='lightblue', s=20, alpha=0.5, label='Train Closed')

# Mark start and end points
ax.plot(latent_path_linear[0, 0], latent_path_linear[0, 1], 
        'ro', c='k', markersize=8, label='Start (Open)')
ax.plot(latent_path_linear[-1, 0], latent_path_linear[-1, 1], 
        'rs', c='k', markersize=8, label='End (Closed)')

ax.set_xlabel('Latent Dimension 1')
ax.set_ylabel('Latent Dimension 2')
ax.set_title('DOPE Surface with Linear and A* Paths')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig(fig_dir / 'path_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Finally, we decode the paths from latent vector to Cartesian coordinates and write to PDBs.

In [ ]:
sample_dir = Path('linear_interpolation_path')
if sample_dir.exists() is False:
    sample_dir.mkdir(parents=True, exist_ok=True)

MA.generate(
    crd = latent_path_linear,
    pdb_path = sample_dir
)

In [ ]:
sample_dir = Path('astar_interpolation_path')
if sample_dir.exists() is False:
    sample_dir.mkdir(parents=True, exist_ok=True)

MA.generate(
    crd = latent_path_astar,
    pdb_path = sample_dir
)

### MolearnGUI

The `MolearnGUI` class provides an **interactive widget-based interface** for exploring trained models directly in Jupyter notebooks. The GUI will automatically detect all datasets registered with the `MolearnAnalysis` object. It provides dropdown menus for dataset selection, visualization, and path generation.

- The GUI is ideal for **quick sanity checks** and exploratory analysis of trained models.
- For custom workflows, use `MolearnAnalysis` methods directly. 
- The interface requires `nglview` and `ipywidgets` packages to be installed.

In [ ]:
import importlib
from molearn.analysis import GUI
importlib.reload(GUI)
from molearn.analysis.GUI import MolearnGUI

In [ ]:
MolearnGUI(MA)